In [1]:
import ee
import geemap
import json

In [2]:
# Initialize the library
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

In [3]:
import geopandas as gpd
# geo_json_string = r'../data/boundaries/blocks_saptari_gee.geojson'
geo_json_string = r'../data/boundaries/districts_nepal_gee.geojson'
df = gpd.read_file(geo_json_string)

In [4]:
df.head(5)

,block_id,fid_1,FIRST_DIST,geometry
0,0,1,ACHHAM,"POLYGON ((81.17111 29.38643, 81.17146 29.38629..."
1,1,2,ARGHAKHANCHI,"POLYGON ((83.0044 28.11188, 83.00485 28.11177,..."
2,2,3,BAGLUNG,"POLYGON ((83.09957 28.63442, 83.10086 28.63401..."
3,3,4,BAITADI,"POLYGON ((80.75833 29.70446, 80.75852 29.70422..."
4,4,5,BAJHANG,"POLYGON ((81.08994 30.05411, 81.09021 30.0541,..."


In [5]:
print(df.crs)

EPSG:4326


In [6]:
# Reproject to WGS 84 (EPSG:4326)
gdf_reprojected = df.to_crs("EPSG:4326")

In [ ]:
gdf_reprojected['geometry'] = gdf_reprojected['geometry'].simplify(tolerance=0.001, preserve_topology=True)
# Convert the GeoDataFrame to a GeoJSON string
geo_json = gdf_reprojected.to_json()

# Create an ee.FeatureCollection from the GeoJSON
ee_feature_collection = ee.FeatureCollection(json.loads(geo_json))

# Now you can use the ee_feature_collection in your GEE scripts
print(f"Successfully converted shapefile to GEE FeatureCollection with {ee_feature_collection.size().getInfo()} features.")

# Example: Print the bounds
print(ee_feature_collection.geometry().bounds().getInfo())

EEException: Request payload size exceeds the limit: 10485760 bytes.

In [21]:
# 1. Setup Assets and Constants
geo_json_string = r'../data/boundaries/blocks_saptari_gee.geojson'
blocks = ee_feature_collection
start_date = '2019-01-01'
end_date = '2024-12-31'

In [22]:
# Initialize Map
Map = geemap.Map()
Map.centerObject(blocks, 7)
Map.addLayer(blocks, {'color': 'red'}, "Saptari Blocks")

In [23]:
Map

Map(center=[26.59775252617876, 86.74855414997488], controls=(WidgetControl(options=['position', 'transparent_b…

In [25]:
# --- 2. Sentinel-2 NDVI ---
def calculate_s2_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi.copyProperties(img, ['system:time_start'])

s2 = (ee.ImageCollection("COPERNICUS/S2_SR")
      .filterBounds(blocks)
      .filterDate(start_date, end_date)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
      .map(calculate_s2_ndvi))

def reduce_s2_regions(img):
    date = ee.Date(img.get('system:time_start'))
    return img.reduceRegions(
        collection=blocks,
        reducer=ee.Reducer.mean(),
        scale=10
    ).map(lambda f: f.set({
        'year': date.get('year'),
        'month': date.get('month')
    }))

ndvi_table_s2 = s2.map(reduce_s2_regions).flatten()

In [31]:
ndvi_table_s2

In [32]:
# Export Sentinel-2 NDVI
task_s2 = ee.batch.Export.table.toDrive(
    collection=ndvi_table_s2,
    folder='EE_Exports',
    description='Saptari_Block_S2_NDVI',
    fileFormat='CSV'
)

In [33]:
task_s2.start() # Uncomment to run

In [34]:
# --- 3. CHIRPS Rainfall ---
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
    .filterBounds(blocks) \
    .filterDate(start_date, end_date)

years = ee.List.sequence(2019, 2024)
months = ee.List.sequence(1, 12)

def create_monthly_rain(y):
    def month_loop(m):
        start = ee.Date.fromYMD(y, m, 1)
        end = start.advance(1, 'month')
        rain = chirps.filterDate(start, end).sum().rename('Rainfall')
        return rain.set({
            'year': y,
            'month': m,
            'system:time_start': start.millis()
        })
    return months.map(month_loop)

monthly_rain = ee.ImageCollection(years.map(create_monthly_rain).flatten())

def reduce_rain_regions(img):
    return img.reduceRegions(
        collection=blocks,
        reducer=ee.Reducer.mean(),
        scale=5000
    ).map(lambda f: f.set({
        'year': img.get('year'),
        'month': img.get('month')
    }))

rain_table = monthly_rain.map(reduce_rain_regions).flatten()

# Export Rainfall
task_rain = ee.batch.Export.table.toDrive(
    collection=rain_table,
    folder='EE_Exports',
    description='Saptari_Block_Rainfall',
    fileFormat='CSV'
)

In [35]:
task_rain.start()

In [36]:
# --- 4. MODIS ET (Evapotranspiration) ---
def process_et(img):
    return img.multiply(0.1).rename('ET').copyProperties(img, ['system:time_start'])

et_col = (ee.ImageCollection("MODIS/061/MOD16A2")
          .filterBounds(blocks)
          .filterDate(start_date, end_date)
          .select('ET')
          .map(process_et))

def reduce_et_regions(img):
    date = ee.Date(img.get('system:time_start'))
    return img.reduceRegions(
        collection=blocks,
        reducer=ee.Reducer.mean(),
        scale=500
    ).map(lambda f: f.set({
        'year': date.get('year'),
        'month': date.get('month')
    }))

et_table = et_col.map(reduce_et_regions).flatten()

# Export ET
task_et = ee.batch.Export.table.toDrive(
    collection=et_table,
    folder='EE_Exports',
    description='Saptari_Block_ET',
    fileFormat='CSV'
)

In [37]:
task_et.start()

In [38]:
# --- 5. MODIS NDVI ---
def process_modis_ndvi(img):
    return img.multiply(0.0001).rename('NDVI').copyProperties(img, ['system:time_start'])

modis_ndvi = (ee.ImageCollection("MODIS/061/MOD13Q1")
              .filterBounds(blocks)
              .filterDate(start_date, end_date)
              .select('NDVI')
              .map(process_modis_ndvi))

def reduce_modis_regions(img):
    date = ee.Date(img.get('system:time_start'))
    return img.reduceRegions(
        collection=blocks,
        reducer=ee.Reducer.mean(),
        scale=250
    ).map(lambda f: f.set({
        'year': date.get('year'),
        'month': date.get('month')
    }))

ndvi_table_modis = modis_ndvi.map(reduce_modis_regions).flatten()

# Export MODIS NDVI
task_modis_ndvi = ee.batch.Export.table.toDrive(
    collection=ndvi_table_modis,
    folder='EE_Exports',
    description='Saptari_Block_MODIS_NDVI',
    fileFormat='CSV'
)

In [39]:
task_modis_ndvi.start()

In [30]:
# Display the map (if using Jupyter)
Map

Map(center=[26.59775252617876, 86.74855414997488], controls=(WidgetControl(options=['position', 'transparent_b…